# Block-level CVAP estimates

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

<div style="text-align: center;"><a class="sd-sphinx-override sd-btn sd-text-wrap sd-btn-primary reference external" href="https://www.dropbox.com/scl/fo/s22x9phl0hldiakn8nbuz/ABKfxHBaak5ra3eBGkNFWMM?rlkey=igpo7qi07oz5tfgjki317o79t&amp;st=gcxkicnc&amp;dl=1">Download tutorial data</a></div>

Citizen voting-age population (CVAP) is a common part of voting and redistricting analysis, but the 
source tables do not publish it at block resolution:

- the decennial PL file publishes voting-age population (VAP) down to the block, but no CVAP at
  all, while
- the ACS publishes CVAP, but only down to the tract.

`block_cvap_estimates()` bridges the gap: it multiplies each block's decennial VAP by its tract's
ACS citizenship rate to estimate block-level CVAP by race.

The example extracts are for the District of Columbia (the smallest block universe), fetched with
`acs_year=2023` and `pl_year=2020` and included in the tutorial `data/census/` directory.

In [ ]:
from pathlib import Path

import pandas as pd

from gerrytools.data import PLBlockVAPTableInfo

census_dir = Path("data/census")
geoid_columns = ["GEOID", "STATEFP", "COUNTYFP", "TRACTCE", "BLOCKCE", "TRACT_GEOID"]
pd.set_option("display.max_columns", None)

## How GerryTools computes CVAP

GerryTools follows the procedure laid out in 
[data-democracy.org/vap-cvap](https://data-democracy.org/vap-cvap). In short, for each race `r` and 
each decennial block `b` in tract `t` of state `s`:

1. **Tract rate.** `tract_rate[t, r] = ACS tract CVAP / ACS tract VAP`, used only when the
   tract's ACS VAP clears `denominator_threshold` (small denominators give noisy rates).
2. **State fallback rate.** `state_rate[s, r] = ACS state CVAP / ACS state VAP`, always defined.
3. **Select** the tract rate when it is trustworthy, otherwise the state rate.
4. **Estimate.** `block_cvap[b, r] = rate * decennial block VAP[b, r]`.

These are estimates produced by an allocation method, not observed block-level CVAP.

## Estimating block CVAP

One call returns one row per decennial block in the state:

```python
import us

from gerrytools.data import block_cvap_estimates

blocks = block_cvap_estimates(us.states.DC, acs_year=2023, pl_year=2020)
```

`pl_year` accepts only `2020` (the vintage whose P3/P4 block variables the underlying
table carries); an unsupported vintage raises before any request is issued.

In [ ]:
blocks = pd.read_csv(
    census_dir / "dc_block_cvap_2023.csv.gz",
    dtype={column: "string" for column in geoid_columns},
)
print("shape:", blocks.shape, "(one row per block)")
list(blocks.columns)

The frame carries three kinds of columns:

- **GEOID components**: `GEOID`, `STATEFP`, `COUNTYFP`, `TRACTCE`, `BLOCKCE`, and `TRACT_GEOID`
  (so you can roll blocks up to their tract),
- **decennial block VAP**: the `{race}_vap_20` inputs from PL tables P3 and P4, and
- **estimated CVAP**: the `{race}_cvap_acs5_23_pl_20` outputs, whose suffix records both source
  vintages.

## Reading a block

For one populated block, here are the decennial inputs next to the estimates:

In [ ]:
sample = blocks[blocks["total_vap_20"] > 0].iloc[0]
comparison = pd.DataFrame(
    {
        "VAP (decennial)": [sample["total_vap_20"], sample["white_vap_20"], sample["black_vap_20"]],
        "CVAP (estimated)": [
            sample["total_cvap_acs5_23_pl_20"],
            sample["white_cvap_acs5_23_pl_20"],
            sample["black_cvap_acs5_23_pl_20"],
        ],
    },
    index=["Total", "White", "Black"],
)
print("block GEOID:", sample["GEOID"])
comparison

## The `denominator_threshold` knob

`denominator_threshold` (default `20`) is the minimum ACS tract VAP required to trust the tract
rate; below it, the block falls back to the statewide rate. Raising it trades variance for bias:
more blocks use the smoother but coarser state rate. Statewide totals barely move, but individual
block estimates shift. Here the default is compared against a strict `500`:

```python
strict = block_cvap_estimates(
    us.states.DC, acs_year=2023, pl_year=2020, denominator_threshold=500
)
```

In [ ]:
strict = pd.read_csv(
    census_dir / "dc_block_cvap_2023_t500.csv.gz",
    dtype={column: "string" for column in geoid_columns},
)

default_black = blocks.set_index("GEOID")["black_cvap_acs5_23_pl_20"]
strict_black = strict.set_index("GEOID")["black_cvap_acs5_23_pl_20"]
changed = (default_black.round(3) != strict_black.round(3)).sum()

print(f"statewide Black CVAP, threshold=20:  {default_black.sum():>10,.0f}")
print(f"statewide Black CVAP, threshold=500: {strict_black.sum():>10,.0f}")
print(f"blocks whose estimate changed: {changed:,} of {len(blocks):,}")

## The underlying table

The race set and the decennial variables come from `PLBlockVAPTableInfo`, a `PLTableInfo` that
wraps PL tables P3 and P4. Its `race_categories` are the groups paired across the decennial VAP
inputs and the ACS CVAP/VAP rates.

In [ ]:
table = PLBlockVAPTableInfo()
print("race categories:", table.race_categories)
print("decennial source columns:", list(table.construct_short_names()))

## Rolling up to a plan

Because the output is block-level data keyed by GEOID, you aggregate it to districts exactly as
you would any block table: attach each block's district assignment, then group and sum the CVAP
columns. Here the roll-up target is the tract (using the provided `TRACT_GEOID`); swapping in a
district assignment is the same `groupby`.

In [ ]:
cvap_columns = [column for column in blocks.columns if "_cvap_" in column]
by_tract = blocks.groupby("TRACT_GEOID")[cvap_columns].sum()
print(f"{len(blocks):,} blocks rolled up to {len(by_tract):,} tracts")
by_tract[
    ["total_cvap_acs5_23_pl_20", "white_cvap_acs5_23_pl_20", "black_cvap_acs5_23_pl_20"]
].head()

## Related

- [ACS guide](acs.ipynb) for the ACS rates this method consumes, and their margins of
  error.
- [Decennial guide](decennial.ipynb) for the block VAP inputs.
- [Overview](overview.ipynb) for GEOID handling, merging, and key setup.
- [Data API](../../api/data.rst)